# FashionMNIST Classifier with PyTorch

A fully connected neural network trained, validated and tested on FashionMNIST.

- Runs on the GPU if one is available, otherwise on the CPU
- Optimizer: SGD, learning rate 0.1
- 20 epochs
- Accuracy tracked with `torchmetrics` after every epoch, then plotted for training and validation

**Colab tip:** for GPU, go to *Runtime → Change runtime type → T4 GPU*.

## 1. Install and import libraries

In [ ]:
# torch and torchvision come preinstalled on Colab; torchmetrics does not
!pip install -q torchmetrics

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torchmetrics
import matplotlib.pyplot as plt

## 2. Select device (GPU if available, otherwise CPU)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA is available. Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

## 3. Hyperparameters

In [ ]:
torch.manual_seed(42)

LEARNING_RATE = 0.1
NUM_EPOCHS = 20
BATCH_SIZE = 64
NUM_CLASSES = 10
VAL_SIZE = 10_000  # held out from the 60,000 training images

## 4. Load FashionMNIST and split into train / validation / test

The official training set (60,000 images) is split into 50,000 for training and 10,000 for validation.
The official test set (10,000 images) is kept for the final evaluation only.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),  # FashionMNIST mean / std
])

full_train_data = datasets.FashionMNIST(root="data", train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root="data", train=False, download=True, transform=transform)

train_size = len(full_train_data) - VAL_SIZE
train_data, val_data = random_split(
    full_train_data, [train_size, VAL_SIZE], generator=torch.Generator().manual_seed(42)
)

pin = device.type == "cuda"
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=pin)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=pin)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=pin)

print(f"Training samples:   {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples:       {len(test_data)}")

## 5. Define the neural network

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)  # raw logits; CrossEntropyLoss applies softmax


model = NeuralNetwork().to(device)
print(model)

## 6. Loss, optimizer and torchmetrics accuracy

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)

# Separate metric objects so training and validation states never mix
train_acc_metric = torchmetrics.Accuracy(task="multiclass", num_classes=NUM_CLASSES).to(device)
val_acc_metric = torchmetrics.Accuracy(task="multiclass", num_classes=NUM_CLASSES).to(device)
test_acc_metric = torchmetrics.Accuracy(task="multiclass", num_classes=NUM_CLASSES).to(device)

## 7. Training and evaluation functions

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, metric):
    model.train()
    metric.reset()
    running_loss = 0.0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        logits = model(X)
        loss = loss_fn(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        metric.update(logits, y)

    return running_loss / len(loader.dataset), metric.compute().item()


@torch.no_grad()
def evaluate(model, loader, loss_fn, metric):
    model.eval()
    metric.reset()
    running_loss = 0.0

    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        running_loss += loss_fn(logits, y).item() * X.size(0)
        metric.update(logits, y)

    return running_loss / len(loader.dataset), metric.compute().item()

## 8. Train for 20 epochs, tracking accuracy after every epoch

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, train_acc_metric)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, val_acc_metric)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch:2d}/{NUM_EPOCHS} | "
        f"train loss {train_loss:.4f}, train acc {train_acc:.4f} | "
        f"val loss {val_loss:.4f}, val acc {val_acc:.4f}"
    )

## 9. Plot training vs. validation accuracy

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_acc"], marker="o", label="Training accuracy")
plt.plot(epochs, history["val_acc"], marker="s", label="Validation accuracy")
plt.title("FashionMNIST: Training vs. Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.xticks(epochs)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 10. Final evaluation on the test set

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, loss_fn, test_acc_metric)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f} ({test_acc * 100:.2f}%)")